In [1]:
import os
from google.colab import drive

# Google Drive'a bağlanın
drive.mount('/content/drive')

Mounted at /content/drive


# ***`MobileNetV2`***

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import precision_recall_fscore_support
import json

# Veri yolları
data_dir = '/content/drive/My Drive/4o insan gpt /ssler-20240820T074747Z-001/ssler/high_ss'
results_file = '/content/drive/My Drive/4o insan gpt /modelsonuc/MobileNetV2.json'

# Resim boyutları ve batch size
img_height = 224
img_width = 224
batch_size = 4

# GPU bellek ayarları
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# Veri yükleyici (data generator) - veri çoğaltma olmadan
train_datagen = ImageDataGenerator(
    rescale=1. / 255,
    validation_split=0.2  # %20 doğrulama verisi
)

# Eğitim ve doğrulama verisi için veri yükleyici
train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',  # Çok sınıflı sınıflandırma
    subset='training',
    classes=['v0', 'v1', 'v2', 'v3', 'v4', 'v5']
)

val_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    classes=['v0', 'v1', 'v2', 'v3', 'v4', 'v5']
)

# Hiperparametreler
activations = ['relu', 'tanh', 'sigmoid']
dropout_rates = [0.0, 0.1, 0.2]
learning_rates = [0.001, 0.01, 0.1]
epochs = 200

en_iyi_accurasi = 0
en_iyi_parametreler = {}
tum_sonuclar = []

# Kullanılacak temel model
temel_modeller = {
    'MobileNetV2': MobileNetV2
}

# Sonuç dosyası kontrolü
if os.path.exists(results_file) and os.path.getsize(results_file) > 0:
    try:
        with open(results_file, 'r') as file:
            tum_sonuclar = json.load(file)
    except json.JSONDecodeError:
        tum_sonuclar = []
else:
    tum_sonuclar = []

# Model eğitimi ve parametre denemeleri
for model_name, base_model_fn in temel_modeller.items():
    for activation in activations:
        for dropout_rate in dropout_rates:
            for learning_rate in learning_rates:
                try:
                    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
                    for layer in base_model.layers:
                        layer.trainable = False

                    x = GlobalAveragePooling2D()(base_model.output)
                    x = Dense(512, activation=activation)(x)
                    x = Dropout(dropout_rate)(x)
                    output = Dense(len(train_generator.class_indices), activation='softmax')(x)

                    model = Model(inputs=base_model.input, outputs=output)
                    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                                  loss='categorical_crossentropy',
                                  metrics=['accuracy'])

                    # Geri çağırmalar (callbacks)
                    early_stopping = EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True)
                    model_checkpoint = ModelCheckpoint('en_iyi_model.keras', monitor='val_accuracy', save_best_only=True)

                    # Model eğitimi
                    history = model.fit(
                        train_generator,
                        steps_per_epoch=train_generator.samples // train_generator.batch_size,
                        epochs=epochs,
                        validation_data=val_generator,
                        validation_steps=val_generator.samples // val_generator.batch_size,
                        callbacks=[early_stopping, model_checkpoint],
                        verbose=0
                    )

                    val_accuracy = history.history['val_accuracy'][-1]
                    num_epochs = len(history.history['loss'])  # Eğitilen epoch sayısı
                    result = {
                        'model': model_name,
                        'activation': activation,
                        'dropout_rate': dropout_rate,
                        'learning_rate': learning_rate,
                        'val_accuracy': val_accuracy,
                        'history': history.history,
                        'num_epochs': num_epochs
                    }
                    tum_sonuclar.append(result)

                    # Tüm denemeleri ekrana yazdırma
                    print(f"Model: {model_name}, Aktivasyon: {activation}, Dropout Oranı: {dropout_rate}, Öğrenme Hızı: {learning_rate}, Doğrulama Başarımı: {val_accuracy}, Epoch Sayısı: {num_epochs}")

                    if val_accuracy > en_iyi_accurasi:
                        en_iyi_accurasi = val_accuracy
                        en_iyi_parametreler = result

                    with open(results_file, 'w') as file:
                        json.dump(tum_sonuclar, file)

                except Exception as e:
                    print(f"Model {model_name}, aktivasyon {activation}, dropout {dropout_rate}, öğrenme hızı {learning_rate} ile hata: {e}")

print("\nEn iyi parametreler:", en_iyi_parametreler)
print("En iyi skor:", en_iyi_accurasi)

# Sonuçları görselleştirme
if en_iyi_parametreler:
    history = en_iyi_parametreler['history']
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.plot(history['loss'], label='Eğitim Kaybı')
    plt.plot(history['val_loss'], label='Doğrulama Kaybı')
    plt.title('Epoch Başına Kayıp')
    plt.xlabel('Epoch')
    plt.ylabel('Kayıp')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['accuracy'], label='Eğitim Başarımı')
    plt.plot(history['val_accuracy'], label='Doğrulama Başarımı')
    plt.title('Epoch Başına Başarım')
    plt.xlabel('Epoch')
    plt.ylabel('Başarım')
    plt.legend()

    plt.show()

    # En iyi modeli yeniden oluşturma
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation=en_iyi_parametreler['activation'])(x)
    x = Dropout(en_iyi_parametreler['dropout_rate'])(x)
    output = Dense(len(train_generator.class_indices), activation='softmax')(x)

    en_iyi_model = Model(inputs=base_model.input, outputs=output)
    en_iyi_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=en_iyi_parametreler['learning_rate']),
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])

    # En iyi modelin ağırlıklarını yükleme
    en_iyi_model.load_weights('en_iyi_model.keras')

    # Tahminler ve metrik hesaplamaları
    predictions = en_iyi_model.predict(val_generator)
    y_true = val_generator.classes
    y_pred = np.argmax(predictions, axis=1)

    precision_weighted, recall_weighted, f1_score_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    precision_micro, recall_micro, f1_score_micro, _ = precision_recall_fscore_support(y_true, y_pred, average='micro')
    precision_macro, recall_macro, f1_score_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    precision_none, recall_none, f1_score_none, _ = precision_recall_fscore_support(y_true, y_pred, average=None)

    print("Ağırlıklı Precision:", precision_weighted)
    print("Ağırlıklı Recall:", recall_weighted)
    print("Ağırlıklı F1 Skoru:", f1_score_weighted)
    print("Micro Precision:", precision_micro)
    print("Micro Recall:", recall_micro)
    print("Micro F1 Skoru:", f1_score_micro)
    print("Macro Precision:", precision_macro)
    print("Macro Recall:", recall_macro)
    print("Macro F1 Skoru:", f1_score_macro)
    print("Precision (Sınıf Bazında):", precision_none)
    print("Recall (Sınıf Bazında):", recall_none)
    print("F1 Skoru (Sınıf Bazında):", f1_score_none)
else:
    print("En iyi parametreler bulunamadı. Model eğitimi başarısız oldu.")


Found 10871 images belonging to 6 classes.
Found 2714 images belonging to 6 classes.
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.0, Öğrenme Hızı: 0.001, Doğrulama Başarımı: 0.0, Epoch Sayısı: 30
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.0, Öğrenme Hızı: 0.01, Doğrulama Başarımı: 0.0, Epoch Sayısı: 24
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.0, Öğrenme Hızı: 0.1, Doğrulama Başarımı: 0.0, Epoch Sayısı: 22
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.1, Öğrenme Hızı: 0.001, Doğrulama Başarımı: 0.0, Epoch Sayısı: 22
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.1, Öğrenme Hızı: 0.01, Doğrulama Başarımı: 0.0, Epoch Sayısı: 22
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.1, Öğrenme Hızı: 0.1, Doğrulama Başarımı: 0.0, Epoch Sayısı: 22
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.2, Öğrenme Hızı: 0.001, Doğrulama Başarımı: 0.0, Epoch Sayısı: 22
Model: MobileNetV2, Aktivasyon: relu, Dropout Oranı: 0.2, Öğrenme Hızı: 0.